In [1]:
from dotenv import load_dotenv
import os 

load_dotenv()
hf_token = os.getenv("HF_TOKEN")

Install Libraries

In [3]:
!pip install -q youtube-transcript-api langchain-community langchain-openai faiss-cpu tiktoken python-dotenv



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [4]:
pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 35.7 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.3
    Uninstalling pip-25.3:
      Successfully uninstalled pip-25.3
Note: you may need to restart the kernel to use updated packages.


In [6]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings , ChatHuggingFace
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

/var/folders/hy/tg467c916lzckn_372ls6ksr0000gp/T/ipykernel_91610/944458453.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Step - 1 : Indexing

In [12]:
# Step 1a -> Indexing (Document Ingestion)

from youtube_transcript_api import YouTubeTranscriptApi


video_id = "29zfXec6HR0" # Only the ID 

api = YouTubeTranscriptApi()

fetched_transcript = api.fetch(video_id)

transcript = " ".join(snippet.text for snippet in fetched_transcript)


In [13]:
fetched_transcript

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='op-eds in the newspapers these days', start=0.08, duration=4.32), FetchedTranscriptSnippet(text="saying that it's all over for India. The", start=2.32, duration=3.6), FetchedTranscriptSnippet(text='India story is finished.', start=4.4, duration=4.08), FetchedTranscriptSnippet(text='>> These are difficult times but we will be', start=5.92, duration=5.04), FetchedTranscriptSnippet(text='able to get over it with the resources', start=8.48, duration=5.039), FetchedTranscriptSnippet(text='that we have and with the kind of uh', start=10.96, duration=5.2), FetchedTranscriptSnippet(text='international commitments and agreements', start=13.519, duration=4.481), FetchedTranscriptSnippet(text='that we are entering and I think Indian', start=16.16, duration=3.92), FetchedTranscriptSnippet(text='economy is in a phase of sustained high', start=18.0, duration=3.84), FetchedTranscriptSnippet(text='growth. So there is going to be high', start=2

In [14]:
# Step 1b -> Indexing (text Splitting)

splitter = RecursiveCharacterTextSplitter(chunk_size = 1000 , chunk_overlap = 200)
chunks = splitter.create_documents([transcript])

In [16]:
len(chunks)

98

In [17]:
chunks[56]

Document(metadata={}, page_content="which are particularly targeted to export promotion. >> States southern states especially are asking for that. >> Well, several states which uh now feel like they can uh particularly after the FDAs. You see entering FDA means suddenly you have a large global market that you can sell to. Now not all states have the wherewithal to take advantage of that. Yeah, BJ keeps saying that that we have these FDAs, take advantage of it, but the manufacturing sector is a little worried because okay, we've signed the FDAs, we know this, but regulatory problems uh that occur as a result of that because Europe has signed it. Yes, but they have very stringent rules and regulations we can't meet. Our products can't meet uh those uh demands, you Well, all of these things happen simultaneously is when you enter an FDA uh there is going to be con continued negotiations on certain metrics but at the same time quality is uh something we are pushing even domestically. Now y

In [18]:
# Step 1c and 1d -> Indexing (Embedding Generation and Storing in vector store)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector_store = FAISS.from_documents(chunks , embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Step 2 : Retrieval

In [19]:
retriever = vector_store.as_retriever(search_type ='similarity' , search_kwargs = {"k" : 4})

In [20]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x1760fecf0>, search_kwargs={'k': 4})

In [21]:
retriever.invoke("What is Shamvika Ravi is talking about ?")


[Document(id='a46cafeb-6f83-44f2-9e98-5e75cf7091b7', metadata={}, page_content="which are coming in and the market [music] solutions. There are many things that come with seemingly unskilled work as well. >> Yes. >> Now that bit is getting done in the market. Call it concage service. >> Yes. [laughter] >> Right. For all segments of the market and and people are willing to pay for it. So who are we to sit on judgment and say this is slavery or whatever. This episode [music] is powered by easebytrip.com. Namaste Jind. You're watching or listening to another edition of the YNI podcast with Smith Prakash. My guest today is really special Shamika Ravi. She's member of the economic advisory council to the prime minister. The conversation today is centered around where the Indian economy is headed. Why is there depleting confidence in Indian markets? How is the government planning ahead if the West Asia war continues indefinitely? And also on demographic changes that Shamika Ravi has spoken e